# A1：Cut-in 加塞场景 —— AEB 算法闭环测试

## 实验目标
- 理解 OpenSCENARIO 场景与 esmini 仿真
- 掌握 `BaseController` 统一接口（`step(observation) → action`）
- 实现一个 TTC 紧急制动算法，通过量化评测

## 通过判据
```python
assert not is_collision and min_ttc >= 1.5
```

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
from common.controller import BaseController
from common.esmini_env import EsminiEnv
print('✓ labs/common 导入成功')

## 1. 实现你的 AEB 控制器
继承 `BaseController`，实现 TTC 分级制动逻辑。

In [ ]:
class MyAEBController(BaseController):
    """TTC 紧急制动控制器（学生实现区）"""

    def __init__(self, config=None):
        super().__init__(config)
        self.ttc_threshold = self.config.get('ttc_threshold', 2.0)

    def step(self, observation: dict) -> dict:
        ego = observation['ego']
        targets = observation.get('targets', [])
        if not targets:
            return {'target_a': 0.5, 'steer_angle': 0.0}

        # 取最近前车，计算 TTC
        lead = min(targets, key=lambda t: abs(t['x'] - ego['x']))
        rel_dist = abs(lead['x'] - ego['x']) - 4.5
        rel_v = ego['v'] - lead['v']

        if rel_v <= 0:
            ttc = float('inf')
        else:
            ttc = rel_dist / rel_v

        # 分级制动
        if ttc < 1.0:
            a = -6.0  # 全力制动
        elif ttc < self.ttc_threshold:
            a = -3.0  # 中度制动
        else:
            a = 0.5   # 正常行驶

        return {'target_a': a, 'steer_angle': 0.0}

print('✓ 控制器定义完成')

## 2. 运行闭环仿真

In [ ]:
# 注意：需要先运行 gen_cutin.py 生成场景文件
# !python3 /home/jovyan/labs/ad2026/gen_cutin.py

controller = MyAEBController(config={'ttc_threshold': 2.0})
env = EsminiEnv(scenario='cutin.xosc', road='cutin.xodr', dt=0.05)

obs = env.reset(ego_x=10.0, ego_v=15.0)
controller.reset()

# 记录轨迹用于评测
trajectory = []
min_ttc = float('inf')
is_collision = False

for step in range(400):  # 20s @ 20Hz
    action = controller.step(obs)
    obs, done, info = env.step(action)
    trajectory.append((obs['sim_time'], obs['ego']['x'], obs['ego']['v']))

    # 计算 TTC（简化版）
    for t in obs.get('targets', []):
        rel_d = abs(t['x'] - obs['ego']['x']) - 4.5
        rel_v = obs['ego']['v'] - t['v']
        if rel_v > 0.1:
            ttc = rel_d / rel_v
            min_ttc = min(min_ttc, ttc)
        if rel_d < 1.0:
            is_collision = True

    if done or is_collision:
        break

env.close()
print(f'仿真结束：步数={len(trajectory)}, 最小TTC={min_ttc:.2f}s, 碰撞={is_collision}')

## 3. 评测断言

In [ ]:
# 通过判据：无碰撞 且 最小TTC ≥ 1.5s
assert not is_collision, '发生碰撞，测试未通过！'
assert min_ttc >= 1.5, f'最小TTC {min_ttc:.2f}s < 1.5s，测试未通过！'
print('✅ 评测通过！')

In [ ]:
# 动画演示：Cut-in 场景示意图（不依赖 esmini，纯 matplotlib）
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

fig, ax = plt.subplots(figsize=(12, 4))
ax.set_xlim(0, 120)
ax.set_ylim(-6, 6)
ax.set_xlabel('纵向位置 (m)')
ax.set_ylabel('横向位置 (m)')
ax.set_title('Cut-in 加塞场景动画演示')
# 画车道线
for y in [-3, 0, 3]:
    ax.axhline(y, color='gray', linestyle='--', alpha=0.5)
ax.text(5, 1.5, 'Ego 车道', fontsize=9)
ax.text(5, -4.5, 'Target 车道', fontsize=9)

# 车辆用矩形表示
from matplotlib.patches import Rectangle
ego_rect = Rectangle((10, -1), 4.5, 2, fc='blue', alpha=0.7, label='Ego')
tgt_rect = Rectangle((45, -4), 4.5, 2, fc='red', alpha=0.7, label='Target')
ax.add_patch(ego_rect)
ax.add_patch(tgt_rect)
ax.legend(loc='upper right')

def animate(frame):
    t = frame * 0.2  # 每帧 0.2s
    # Ego: 15m/s 匀速
    ego_x = 10 + 15 * t
    # Target: 12m/s，t=5s 开始变道（3s 完成）
    tgt_x = 45 + 12 * t
    if t < 5:
        tgt_y = -3
    elif t < 8:
        progress = (t - 5) / 3
        tgt_y = -3 + 3 * np.sin(progress * np.pi / 2)  # 平滑变道
    else:
        tgt_y = 0
    ego_rect.set_xy((ego_x, -1))
    tgt_rect.set_xy((tgt_x, tgt_y - 1))
    ax.set_title(f'Cut-in 场景 t={t:.1f}s')
    return ego_rect, tgt_rect

ani = animation.FuncAnimation(fig, animate, frames=60, interval=200, blit=False)
plt.close()  # 防止静态显示
from IPython.display import HTML
HTML(ani.to_jshtml())


## 5. 运行结果可视化

In [ ]:
import matplotlib.pyplot as plt

times = [t for t, x, v in trajectory]
speeds = [v for t, x, v in trajectory]

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(times, speeds)
plt.xlabel('时间 (s)'); plt.ylabel('自车速度 (m/s)')
plt.title('AEB 制动过程速度曲线')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot([x for t, x, v in trajectory], speeds)
plt.xlabel('位置 (m)'); plt.ylabel('速度 (m/s)')
plt.title('位置-速度轨迹')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. 🏆 挑战任务

1. **调参挑战**：把 `ttc_threshold` 从 2.0 改为 1.5 和 2.5，各跑一次。哪个更安全？哪个更舒适？（提示：看最大减速度）
2. **算法挑战**：实现一个考虑前车加速度的改进版 TTC（提示：`rel_v` 用相对加速度修正），看最小 TTC 能否提升。
3. **场景挑战**：修改 `gen_cutin.py`，让 Target 以 18m/s 切入（比 Ego 快），你的 AEB 还会误触发吗？